In [2]:
 
# LIGHTCONE GENERATION - BOX SIZE SCAN
# py21cmfast v3.4
# Fixed Resolution (HII_DIM), varying BOX_LEN from 400 to 2000 Mpc
# =============================================================================

# =============================================================================
# CELL 1: Imports and Setup
# =============================================================================

import numpy as np
import matplotlib as mpl

import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import matplotlib.pyplot as plt

import py21cmfast as p21c
from py21cmfast import plotting

import os
from datetime import datetime

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Create Output Directory for Plots
# =============================================================================

plot_dir = "BOX_SIZE_SCAN_13DEC2025/plots"

# Create directory if it doesn't exist
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")

print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# =============================================================================
# CELL 1b: Standardized Plot Settings
# =============================================================================

plt.rcParams.update({
    # Font settings
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 20,
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 14,
    'figure.titlesize': 20,
    
    # Professional ticks
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.top': True,
    'ytick.right': True,
    
    # Line and axes
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    
    # Figure
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

print("✓ Plot settings applied")

# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================

# **FIXED: RESOLUTION**
HII_DIM_FIXED = 128

# **SCAN: BOX_LEN (400 to 2000 Mpc in steps of 400)**
BOX_LEN_VALUES = np.arange(400, 2001, 400)  # [400, 800, 1200, 1600, 2000] Mpc

# Redshift range for the lightcone
z_min = 5.0
z_max = 20.0

print(f"\n=== PARAMETER SCAN SETUP ===")
print(f"Fixed HII_DIM = {HII_DIM_FIXED}")
print(f"\nScanning BOX_LEN:")
print(f"  Number of values: {len(BOX_LEN_VALUES)}")
print(f"  Range: {BOX_LEN_VALUES.min():.0f} Mpc → {BOX_LEN_VALUES.max():.0f} Mpc")
print(f"  Step size: {BOX_LEN_VALUES[1] - BOX_LEN_VALUES[0]:.0f} Mpc")
print(f"  Values: {BOX_LEN_VALUES}")
print(f"\nTotal simulations: {len(BOX_LEN_VALUES)}")

print(f"\nRedshift range: z = {z_min} → {z_max}")

print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())

print("\n=== DEFAULT ASTROPHYSICS (REIONIZATION KEPT DEFAULT) ===")
default_astro = p21c.AstroParams()
print(default_astro)

print("\n=== DEFAULT FLAGS ===")
print(p21c.FlagOptions())

# Print resolution info for each box size
print("\n=== RESOLUTION DETAILS ===")
print(f"{'BOX_LEN [Mpc]':<15} {'HII_DIM':<10} {'Cell Size [Mpc]':<20} {'Cell Size [kpc]':<15}")
print("-" * 70)
for box_len in BOX_LEN_VALUES:
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    print(f"{box_len:<15.0f} {HII_DIM_FIXED:<10} {cell_size_mpc:<20.3f} {cell_size_kpc:<15.1f}")



/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:57: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_cfg.py:41: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn("Your configuration file is out of date. Updating...")


py21cmfast version: 3.3.1
Created directory: BOX_SIZE_SCAN_13DEC2025/plots
All plots will be saved to: /user1/swanith/BOX_SIZE_SCAN_13DEC2025/plots
✓ Plot settings applied

=== PARAMETER SCAN SETUP ===
Fixed HII_DIM = 128

Scanning BOX_LEN:
  Number of values: 5
  Range: 400 Mpc → 2000 Mpc
  Step size: 400 Mpc
  Values: [ 400  800 1200 1600 2000]

Total simulations: 5

Redshift range: z = 5.0 → 20.0

=== DEFAULT COSMOLOGY ===
CosmoParams:
    OMb        : 0.04897468161869667
    OMm        : 0.30964144154550644
    POWER_INDEX: 0.9665
    SIGMA_8    : 0.8102
    hlittle    : 0.6766
    

=== DEFAULT ASTROPHYSICS (REIONIZATION KEPT DEFAULT) ===
AstroParams:
    ALPHA_ESC       : -0.5
    ALPHA_STAR      : 0.5
    ALPHA_STAR_MINI : 0.5
    A_LW            : 2.0
    A_VCB           : 1.0
    BETA_LW         : 0.6
    BETA_VCB        : 1.8
    F_ESC10         : 0.1
    F_ESC7_MINI     : 0.01
    F_H2_SHIELD     : 0.0
    F_STAR10        : 0.05011872336272722
    F_STAR7_MINI    : 0.01
    

In [3]:
# =============================================================================
# CELL 2: Run Lightcone Simulations - BOX_LEN Scan
# =============================================================================

import time

print("\n" + "="*70)
print("RUNNING BOX_LEN SCAN")
print("="*70)

# Create main cache directory
main_cache_dir = "BOX_SIZE_SCAN_13DEC2025/cache"
if not os.path.exists(main_cache_dir):
    os.makedirs(main_cache_dir)
    print(f"Created cache directory: {main_cache_dir}")
else:
    print(f"Cache directory exists: {main_cache_dir}")

# Dictionary to store lightcone results
lightcones = {}

# Track timing
scan_start_time = time.time()

for idx, box_len in enumerate(BOX_LEN_VALUES):
    sim_start_time = time.time()
    
    print(f"\n{'='*70}")
    print(f"SIMULATION {idx+1}/{len(BOX_LEN_VALUES)}")
    print(f"BOX_LEN = {box_len:.0f} Mpc")
    print(f"HII_DIM = {HII_DIM_FIXED}")
    print(f"Cell size = {box_len/HII_DIM_FIXED:.3f} Mpc = {box_len/HII_DIM_FIXED*1000:.1f} kpc")
    print(f"{'='*70}")
    
    # Define user parameters for this box size
    user_params = p21c.UserParams(
        HII_DIM=HII_DIM_FIXED,
        BOX_LEN=box_len,
        USE_INTERPOLATION_TABLES=True,
        N_THREADS=8
    )
    
    print(f"Redshift range: z = {z_min} → {z_max}")
    print(f"Resolution: {user_params.HII_DIM}³ cells")
    
    # Create subdirectory for this box size
    cache_subdir = f"{main_cache_dir}/BOX{box_len:.0f}_DIM{HII_DIM_FIXED}"
    
    # Run lightcone simulation (with default astrophysics)
    try:
        lightcone = p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
            user_params=user_params,
            random_seed=37,
            direc=cache_subdir
        )
        
        # Store the lightcone
        lightcones[box_len] = lightcone
        
        sim_time = time.time() - sim_start_time
        
        print(f"\n✓ Simulation complete!")
        print(f"  Time: {sim_time/60:.2f} minutes")
        print(f"  Cache: {cache_subdir}")
        print(f"  Shape: {lightcone.brightness_temp.shape}")
        print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
              f"{lightcone.lightcone_redshifts.max():.2f}]")
        
        # Quick reionization stats
        z_nodes = lightcone.node_redshifts[::-1]
        x_e_nodes = 1.0 - lightcone.global_xH[::-1]
        
        try:
            idx_10 = np.argmin(np.abs(x_e_nodes - 0.1))
            idx_50 = np.argmin(np.abs(x_e_nodes - 0.5))
            idx_90 = np.argmin(np.abs(x_e_nodes - 0.9))
            
            z_10 = z_nodes[idx_10]
            z_50 = z_nodes[idx_50]
            z_90 = z_nodes[idx_90]
            delta_z = z_10 - z_90
            
            print(f"  Quick stats:")
            print(f"    z(10% ionized) = {z_10:.2f}")
            print(f"    z(50% ionized) = {z_50:.2f}")
            print(f"    z(90% ionized) = {z_90:.2f}")
            print(f"    Δz (10%→90%) = {delta_z:.2f}")
        except:
            print(f"  Could not compute reionization stats")
        
        # Progress estimate
        elapsed = time.time() - scan_start_time
        avg_time_per_sim = elapsed / (idx + 1)
        remaining_sims = len(BOX_LEN_VALUES) - (idx + 1)
        eta_minutes = (remaining_sims * avg_time_per_sim) / 60
        
        print(f"\n  Progress: {idx+1}/{len(BOX_LEN_VALUES)} ({100*(idx+1)/len(BOX_LEN_VALUES):.1f}%)")
        print(f"  Average time per sim: {avg_time_per_sim/60:.2f} min")
        print(f"  ETA: {eta_minutes:.1f} minutes (~{eta_minutes/60:.2f} hours)")
        
    except Exception as e:
        print(f"\n✗ Simulation FAILED!")
        print(f"  Error: {e}")
        lightcones[box_len] = None

total_time = time.time() - scan_start_time

print(f"\n{'='*70}")
print("ALL SIMULATIONS COMPLETE")
print(f"Total time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
print(f"Successful simulations: {sum(1 for lc in lightcones.values() if lc is not None)}/{len(BOX_LEN_VALUES)}")
print("="*70)



RUNNING BOX_LEN SCAN
Created cache directory: BOX_SIZE_SCAN_13DEC2025/cache

SIMULATION 1/5
BOX_LEN = 400 Mpc
HII_DIM = 128
Cell size = 3.125 Mpc = 3125.0 kpc
Redshift range: z = 5.0 → 20.0
Resolution: 128³ cells


/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:400: UserWarning: The following parameters to FlagOptions are not supported: ['USE_VELS_AUX']
  warnings.warn(
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vx
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vy
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to remove array that isn't yet created: hires_vz
  warnings.warn(f"Trying to remove array that isn't yet created: {k}")
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/py21cmfast/_utils.py:815: UserWarning: Trying to r


✓ Simulation complete!
  Time: 1.82 minutes
  Cache: BOX_SIZE_SCAN_13DEC2025/cache/BOX400_DIM128
  Shape: (128, 128, 965)
  Redshift range: [5.00, 20.06]
  Quick stats:
    z(10% ionized) = 10.76
    z(50% ionized) = 7.74
    z(90% ionized) = 6.17
    Δz (10%→90%) = 4.59

  Progress: 1/5 (20.0%)
  Average time per sim: 1.91 min
  ETA: 7.7 minutes (~0.13 hours)

SIMULATION 2/5
BOX_LEN = 800 Mpc
HII_DIM = 128
Cell size = 6.250 Mpc = 6250.0 kpc
Redshift range: z = 5.0 → 20.0
Resolution: 128³ cells

✓ Simulation complete!
  Time: 1.65 minutes
  Cache: BOX_SIZE_SCAN_13DEC2025/cache/BOX800_DIM128
  Shape: (128, 128, 483)
  Redshift range: [5.00, 20.10]
  Quick stats:
    z(10% ionized) = 11.00
    z(50% ionized) = 8.09
    z(90% ionized) = 6.31
    Δz (10%→90%) = 4.69

  Progress: 2/5 (40.0%)
  Average time per sim: 1.80 min
  ETA: 5.4 minutes (~0.09 hours)

SIMULATION 3/5
BOX_LEN = 1200 Mpc
HII_DIM = 128
Cell size = 9.375 Mpc = 9375.0 kpc
Redshift range: z = 5.0 → 20.0
Resolution: 128³ cel

In [4]:
# =============================================================================
# CELL 2b: Plot Lightcones (All Box Sizes Stacked)
# =============================================================================
print("\n" + "="*70)
print("GENERATING STACKED LIGHTCONE PLOTS")
print("="*70)

# We'll plot all box sizes in the scan
print(f"Plotting all {len(BOX_LEN_VALUES)} box sizes")

# Create colormap for labeling
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# Fields to plot
fields_to_plot = [
    ('brightness_temp', '21cm Brightness Temperature', 'EoR'),
    ('xH_box', 'Neutral Fraction (xHI)', 'viridis'),
    ('density', 'Overdensity δ', 'magma'),
    ('velocity', 'Line-of-Sight Velocity', 'RdBu_r')
]

for field_name, field_title, field_cmap in fields_to_plot:
    print(f"\nPlotting {field_name}...")
    
    # Create figure with one subplot per box size
    fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                            figsize=(14, 4*len(BOX_LEN_VALUES)), 
                            constrained_layout=True)
    
    if len(BOX_LEN_VALUES) == 1:
        axes = [axes]
    
    for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
        if box_len not in lightcones or lightcones[box_len] is None:
            ax.text(0.5, 0.5, f'BOX_LEN = {box_len:.0f} Mpc\nSimulation Failed', 
                   ha='center', va='center', fontsize=16, color='red',
                   transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
            
        lightcone = lightcones[box_len]
        
        # Plot using py21cmfast's built-in plotter
        plotting.lightcone_sliceplot(lightcone, field_name, ax=ax, fig=fig)
        
        # Change colormap
        im = ax.images[0]
        im.set_cmap(field_cmap)
        
        # Get color for this box size
        color = cmap(norm(box_len))
        
        # Calculate cell size
        cell_size_mpc = box_len / HII_DIM_FIXED
        cell_size_kpc = cell_size_mpc * 1000
        
        # Add label with box info
        ax.text(0.02, 0.98, 
               f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.1f} kpc)', 
               transform=ax.transAxes, fontsize=13, fontweight='bold',
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.7, 
                        edgecolor='black', linewidth=2))
        
        # Add box size on right side as well
        ax.text(0.98, 0.98, 
               f'{box_len:.0f} Mpc', 
               transform=ax.transAxes, fontsize=14, fontweight='bold',
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Save
    plot_name = f"{field_name}_lightcone_boxsize_stack"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    
    fig.suptitle(f'{field_title} - Box Size Scan (HII_DIM={HII_DIM_FIXED})', 
                fontsize=22, fontweight='bold', y=0.995)
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    plt.close(fig)

print("\n✓ STACKED LIGHTCONE PLOTTING COMPLETE!")


GENERATING STACKED LIGHTCONE PLOTS
Plotting all 5 box sizes

Plotting brightness_temp...
  ✓ Saved: brightness_temp_lightcone_boxsize_stack

Plotting xH_box...
  ✓ Saved: xH_box_lightcone_boxsize_stack

Plotting density...
  ✓ Saved: density_lightcone_boxsize_stack

Plotting velocity...
  ✓ Saved: velocity_lightcone_boxsize_stack

✓ STACKED LIGHTCONE PLOTTING COMPLETE!


In [5]:
# =============================================================================
# CELL 3: 2D SLICES AT z = 8 (ALL BOX SIZES)
# =============================================================================

print("\n" + "="*70)
print("GENERATING 2D SLICES AT z = 8.0")
print("="*70)

target_z = 8.0

# Use all box sizes
print(f"Using all {len(BOX_LEN_VALUES)} BOX_LEN values for slices")

fields_info = [
    ('brightness_temp', '21cm Brightness Temperature [mK]', 'EoR'),
    ('xH_box', 'Neutral Fraction (xHI)', 'viridis'),
    ('density', 'Overdensity δ', 'magma'),
    ('velocity', 'Line-of-Sight Velocity [km/s]', 'RdBu_r'),
]

for field_name, field_label, cmap_name in fields_info:
    print(f"\nProcessing {field_name}...")
    
    # Create figure with all box sizes
    fig, axes = plt.subplots(1, len(BOX_LEN_VALUES), 
                            figsize=(5*len(BOX_LEN_VALUES), 5.5), 
                            constrained_layout=True)
    
    if len(BOX_LEN_VALUES) == 1:
        axes = [axes]
    
    # First pass: find global min/max for consistent colorbar
    vmin_global = np.inf
    vmax_global = -np.inf
    slices_data = []
    
    for box_len in BOX_LEN_VALUES:
        if box_len not in lightcones or lightcones[box_len] is None:
            slices_data.append((None, None, box_len))
            continue
            
        lightcone = lightcones[box_len]
        z_values = lightcone.lightcone_redshifts
        closest_idx = np.argmin(np.abs(z_values - target_z))
        actual_z = z_values[closest_idx]
        
        field_data = getattr(lightcone, field_name)
        slice_2d = field_data[:, :, closest_idx]
        
        slices_data.append((slice_2d, actual_z, box_len))
        
        vmin_global = min(vmin_global, slice_2d.min())
        vmax_global = max(vmax_global, slice_2d.max())
    
    # Create rainbow colormap for box labels
    cmap_rainbow = mpl.cm.rainbow
    norm_rainbow = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())
    
    # Second pass: plot with consistent colorbar
    for idx, (data_tuple, ax) in enumerate(zip(slices_data, axes)):
        slice_2d, actual_z, box_len = data_tuple
        
        if slice_2d is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                   transform=ax.transAxes, fontsize=16)
            ax.set_title(f'BOX={box_len:.0f} Mpc', fontsize=14)
            continue
        
        im = ax.imshow(slice_2d.T, origin='lower', cmap=cmap_name,
                      vmin=vmin_global, vmax=vmax_global,
                      extent=[0, box_len, 0, box_len],
                      aspect='auto')
        
        ax.set_xlabel('Comoving Distance [Mpc]', fontsize=12)
        ax.set_ylabel('Comoving Distance [Mpc]', fontsize=12)
        
        # Get rainbow color for this box size
        color_label = cmap_rainbow(norm_rainbow(box_len))
        
        # Calculate cell size
        cell_size_mpc = box_len / HII_DIM_FIXED
        cell_size_kpc = cell_size_mpc * 1000
        
        ax.set_title(f'BOX={box_len:.0f} Mpc\nCell={cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)\nz={actual_z:.2f}', 
                    fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.6, 
                             edgecolor='black', linewidth=1.5))
        
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)
    
    # Overall title
    fig.suptitle(f'{field_label} at z ≈ {target_z} (HII_DIM={HII_DIM_FIXED})', 
                fontsize=18, fontweight='bold')
    
    # Save
    plot_name = f"{field_name}_slice_z{int(target_z)}_boxsize_all"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    print(f"    Range: [{vmin_global:.3f}, {vmax_global:.3f}]")
    
    plt.close(fig)

print("\n✓ 2D SLICE PLOTTING COMPLETE!")

# =============================================================================
# CELL 3b: Print Statistics for 2D Slices
# =============================================================================

print("\n" + "="*70)
print("2D SLICE STATISTICS AT z = 8")
print("="*70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    z_values = lightcone.lightcone_redshifts
    closest_idx = np.argmin(np.abs(z_values - target_z))
    actual_z = z_values[closest_idx]
    
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\nBOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc) at z = {actual_z:.2f}:")
    
    for field_name, field_label, _ in fields_info:
        field_data = getattr(lightcone, field_name)
        slice_2d = field_data[:, :, closest_idx]
        
        print(f"  {field_name:15s}: min={slice_2d.min():10.3e}, "
              f"max={slice_2d.max():10.3e}, mean={slice_2d.mean():10.3e}")

print("\n" + "="*70)

# =============================================================================
# CELL 3c: Summary Statistics for 2D Slices at z=8
# =============================================================================

print("\n" + "="*70)
print(f"SUMMARY STATISTICS FOR 2D SLICES AT z ≈ {target_z}")
print("="*70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    
    # Find the closest redshift slice
    z_values = lightcone.lightcone_redshifts
    closest_idx = np.argmin(np.abs(z_values - target_z))
    actual_z = z_values[closest_idx]
    
    # Extract 2D slices
    brightness_slice = lightcone.brightness_temp[:, :, closest_idx]
    xHI_slice = lightcone.xH_box[:, :, closest_idx]
    density_slice = lightcone.density[:, :, closest_idx]  # δ
    velocity_slice = lightcone.velocity[:, :, closest_idx]
    
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\nBOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc) at z = {actual_z:.2f}:")
    print(f"  Brightness temp [mK]: min={brightness_slice.min():.2f}, "
          f"max={brightness_slice.max():.2f}, mean={brightness_slice.mean():.2f}")
    print(f"  Neutral fraction:     min={xHI_slice.min():.4f}, "
          f"max={xHI_slice.max():.4f}, mean={xHI_slice.mean():.4f}")
    print(f"  Overdensity δ:        min={density_slice.min():.3f}, "
          f"max={density_slice.max():.3f}, mean={density_slice.mean():.3f}")
    print(f"  Velocity [km/s]:      min={velocity_slice.min():.2f}, "
          f"max={velocity_slice.max():.2f}, mean={velocity_slice.mean():.2f}")

print("\n" + "="*70)

# =============================================================================
# CELL 3d: Summary Statistics for Full Lightcones (All Box Sizes)
# =============================================================================

print("\n" + "="*70)
print("SUMMARY STATISTICS FOR FULL LIGHTCONES (ALL BOX SIZES)")
print("="*70)
print(f"Fixed: HII_DIM = {HII_DIM_FIXED}")
print(f"Total simulations: {len(BOX_LEN_VALUES)}")

for idx, box_len in enumerate(BOX_LEN_VALUES):
    if box_len not in lightcones or lightcones[box_len] is None:
        print(f"\n[{idx+1}/{len(BOX_LEN_VALUES)}] BOX_LEN = {box_len:.0f} Mpc: FAILED")
        continue
        
    lightcone = lightcones[box_len]
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\n[{idx+1}/{len(BOX_LEN_VALUES)}] BOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc):")
    print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
          f"{lightcone.lightcone_redshifts.max():.2f}]")
    
    # Full 3D field statistics
    print(f"  Brightness temp [mK]: min={lightcone.brightness_temp.min():.2f}, "
          f"max={lightcone.brightness_temp.max():.2f}, "
          f"mean={lightcone.brightness_temp.mean():.2f}")
    print(f"  Neutral fraction:     min={lightcone.xH_box.min():.4f}, "
          f"max={lightcone.xH_box.max():.4f}, "
          f"mean={lightcone.xH_box.mean():.4f}")
    print(f"  Overdensity δ:        min={lightcone.density.min():.3f}, "
          f"max={lightcone.density.max():.3f}, "
          f"mean={lightcone.density.mean():.3f}")
    print(f"  Velocity [km/s]:      min={lightcone.velocity.min():.2f}, "
          f"max={lightcone.velocity.max():.2f}, "
          f"mean={lightcone.velocity.mean():.2f}")

print("\n" + "="*70)

# =============================================================================
# CELL 3e: Compact Summary Table
# =============================================================================

print("\n" + "="*70)
print("COMPACT SUMMARY: REIONIZATION PROGRESS BY BOX SIZE")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'<xHI>':<10} {'<Tb>[mK]':<12} {'z_range':<15}")
print("-" * 70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    mean_xHI = lightcone.xH_box.mean()
    mean_Tb = lightcone.brightness_temp.mean()
    z_min_actual = lightcone.lightcone_redshifts.min()
    z_max_actual = lightcone.lightcone_redshifts.max()
    
    print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} {mean_xHI:<10.4f} {mean_Tb:<12.2f} "
          f"[{z_min_actual:.2f}, {z_max_actual:.2f}]")

print("="*70)


GENERATING 2D SLICES AT z = 8.0
Using all 5 BOX_LEN values for slices

Processing brightness_temp...
  ✓ Saved: brightness_temp_slice_z8_boxsize_all
    Range: [0.000, 25.470]

Processing xH_box...
  ✓ Saved: xH_box_slice_z8_boxsize_all
    Range: [0.000, 1.000]

Processing density...
  ✓ Saved: density_slice_z8_boxsize_all
    Range: [-0.536, 1.827]

Processing velocity...
  ✓ Saved: velocity_slice_z8_boxsize_all
    Range: [-0.000, 0.000]

✓ 2D SLICE PLOTTING COMPLETE!

2D SLICE STATISTICS AT z = 8

BOX_LEN = 400 Mpc (Cell = 3.12 Mpc = 3125 kpc) at z = 8.00:
  brightness_temp: min= 0.000e+00, max= 2.547e+01, mean= 1.202e+01
  xH_box         : min= 0.000e+00, max= 1.000e+00, mean= 5.457e-01
  density        : min=-5.358e-01, max= 1.827e+00, mean= 6.200e-03
  velocity       : min=-1.198e-16, max= 1.027e-16, mean= 7.610e-18

BOX_LEN = 800 Mpc (Cell = 6.25 Mpc = 6250 kpc) at z = 7.99:
  brightness_temp: min= 0.000e+00, max= 1.879e+01, mean= 1.126e+01
  xH_box         : min= 0.000e+00, m

In [6]:
# =============================================================================
# CELL 4: Reionization History Analysis - Box Size Scan
# =============================================================================
print("\n" + "="*70)
print("GENERATING REIONIZATION HISTORY COMPARISON")
print("="*70)

# Create rainbow colormap for all box sizes
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# =============================================================================
# PLOT 4a: Ionization Fraction vs Redshift (All Box Sizes)
# =============================================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    z_nodes = lightcone.node_redshifts[::-1]
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    color = cmap(norm(box_len))
    
    ax.plot(z_nodes, x_e_nodes, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Ionization Fraction $x_e$', fontsize=20)
ax.set_ylim(-0.05, 1.05)
ax.invert_xaxis()
ax.legend(fontsize=12, loc='best', ncol=1)
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "reionization_history_xe_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'Reionization History: Ionization Fraction (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 4b: Neutral Fraction vs Redshift (All Box Sizes)
# =============================================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    z_nodes = lightcone.node_redshifts[::-1]
    xHI_nodes = lightcone.global_xH[::-1]
    
    color = cmap(norm(box_len))
    
    ax.plot(z_nodes, xHI_nodes, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Neutral Fraction $x_{\rm HI}$', fontsize=20)
ax.set_ylim(-0.05, 1.05)
ax.invert_xaxis()
ax.legend(fontsize=12, loc='best', ncol=1)
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "reionization_history_xHI_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'Reionization History: Neutral Fraction (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# CELL 4f: Optical Depth Calculations for All Box Sizes
# =============================================================================

print("\n" + "="*70)
print("OPTICAL DEPTH CALCULATIONS")
print("="*70)

# Physical constants for optical depth calculation
c_km_s = 2.998e5                    # Speed of light [km/s]
h = 0.6766                          # Hubble parameter (from default cosmology)
H0 = 100 * h                        # Hubble constant [km/s/Mpc]
Omega_b = 0.04897468161869667       # Baryon density (from default cosmology)
Omega_m = 0.30964144154550644       # Matter density (from default cosmology)

# Critical density of the universe [protons/cm^3]
rho_crit_p_cm3 = 1.88e-29 * h**2 / (1.67e-24)  # Convert to protons/cm^3
n_H0_cm3 = Omega_b * rho_crit_p_cm3             # Mean hydrogen number density [cm^-3]

# Thomson scattering cross section
sigma_T_cm2 = 6.65e-25              # [cm^2]

# Convert to Mpc units
cm_per_Mpc = 3.086e24
n_e0_Mpc3 = n_H0_cm3 * cm_per_Mpc**3
sigma_T_Mpc2 = sigma_T_cm2 / cm_per_Mpc**2

# Prefactor for dτ calculation
prefactor = n_e0_Mpc3 * sigma_T_Mpc2  # [Mpc^-1]

print(f"\nPhysical constants:")
print(f"  n_H0 = {n_H0_cm3:.6e} cm^-3")
print(f"  σ_T = {sigma_T_cm2:.6e} cm^2")
print(f"  Prefactor = {prefactor:.6e} Mpc^-1")

# Dictionary to store optical depth results
tau_results = {}

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    
    # Extract redshift and distance axes
    red_axis = lightcone.lightcone_redshifts
    pos_axis = lightcone.lightcone_distances  # Comoving distance [Mpc]
    
    # Trim to z <= z_max if needed
    ind_z = np.where(red_axis <= z_max)[0]
    red_axis = red_axis[ind_z]
    pos_axis = pos_axis[ind_z]
    
    # Get ionization history
    z_nodes_sorted = lightcone.node_redshifts[::-1]
    xHI_nodes_sorted = lightcone.global_xH[::-1]
    x_e_nodes_sorted = 1.0 - xHI_nodes_sorted
    
    # Interpolate x_e onto lightcone redshift grid
    x_e_interp = np.interp(red_axis, z_nodes_sorted, x_e_nodes_sorted)
    
    # Calculate geometric quantities
    s = pos_axis  # Comoving distance [Mpc]
    ds = np.diff(s)  # Distance element [Mpc]
    
    # Midpoint values for integration
    z_mid = 0.5 * (red_axis[:-1] + red_axis[1:])
    x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])
    
    # Calculate dτ
    dtau = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds
    
    # Cumulative optical depth
    tau = np.cumsum(dtau)
    tau_total = tau[-1]
    
    # Store results
    tau_results[box_len] = {
        'red_axis': red_axis,
        'pos_axis': pos_axis,
        's': s,
        'ds': ds,
        'z_mid': z_mid,
        'x_e_interp': x_e_interp,
        'x_e_mid': x_e_mid,
        'dtau': dtau,
        'tau': tau,
        'tau_total': tau_total
    }

# Print compact summary
print(f"\n{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'z_range':<15} {'<ds>[Mpc]':<12} {'τ_total':<10}")
print("-" * 75)

for box_len in sorted(tau_results.keys()):
    results = tau_results[box_len]
    red_axis = np.asarray(results['red_axis'])
    ds = np.asarray(results['ds'])
    tau_total = float(np.asarray(results['tau_total']))
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    
    print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} [{red_axis.min():.2f}, {red_axis.max():.2f}] "
          f"{ds.mean():<12.3f} {tau_total:<10.6f}")

print("\n" + "="*70)

# =============================================================================
# PLOT: Cumulative Optical Depth τ vs z (All Box Sizes - Rainbow)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(tau_results.keys()):
    results = tau_results[box_len]
    
    z_mid_plot = np.asarray(results['z_mid'])
    tau_plot = np.asarray(results['tau'])
    
    color = cmap(norm(box_len))
    
    ax.plot(z_mid_plot, tau_plot, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$', fontsize=20)
ax.invert_xaxis()
ax.legend(fontsize=12, loc='best', ncol=1)
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "tau_vs_z_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'Cumulative Optical Depth vs Redshift (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Optical Depth Element dτ vs z (All Box Sizes - Rainbow)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(tau_results.keys()):
    results = tau_results[box_len]
    z_mid_plot = np.asarray(results['z_mid'])
    dtau_plot = np.asarray(results['dtau'])
    
    color = cmap(norm(box_len))
    
    ax.plot(z_mid_plot, dtau_plot, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Optical Depth Element $d\tau$', fontsize=20)
ax.legend(fontsize=12, loc='best', ncol=1)
ax.invert_xaxis()
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "dtau_vs_z_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Optical Depth Element $d\tau$ vs Redshift', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Total Optical Depth vs BOX_LEN
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

box_vals = sorted(tau_results.keys())
tau_total_vals = [float(np.asarray(tau_results[b]['tau_total'])) for b in box_vals]
cell_sizes_kpc = [(b / HII_DIM_FIXED) * 1000 for b in box_vals]

ax.plot(box_vals, tau_total_vals, 'o-', linewidth=3, markersize=10,
       color='darkblue', label=r'Total $\tau$')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel(r'Total Optical Depth $\tau$', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "tau_total_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title('Total Optical Depth vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Total Optical Depth vs Cell Size
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

ax.plot(cell_sizes_kpc, tau_total_vals, 's-', linewidth=3, markersize=10,
       color='darkred', label=r'Total $\tau$')

ax.set_xlabel(r'Cell Size [kpc]', fontsize=20)
ax.set_ylabel(r'Total Optical Depth $\tau$', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "tau_total_vs_cellsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title('Total Optical Depth vs Cell Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Comoving Distance s vs z (Overlay - should be identical)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

# Plot all box sizes to verify they're the same
for box_len in sorted(tau_results.keys()):
    results = tau_results[box_len]
    red_axis_plot = np.asarray(results['red_axis'])
    s_plot = np.asarray(results['s'])
    
    color = cmap(norm(box_len))
    
    ax.plot(red_axis_plot, s_plot, 
           linewidth=2.0, color=color, alpha=0.7,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel('Comoving Distance $s$ [Mpc]', fontsize=20)
ax.legend(fontsize=12, loc='best')
ax.invert_xaxis()
ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "s_vs_z_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title('Comoving Distance vs Redshift (Independent of Box Size)', 
            fontsize=18, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n" + "="*70)
print("OPTICAL DEPTH ANALYSIS COMPLETE!")
if len(tau_total_vals) > 0:
    print(f"τ range: {min(tau_total_vals):.6f} → {max(tau_total_vals):.6f}")
    if min(tau_total_vals) > 0:
        print(f"Variation: {(max(tau_total_vals) - min(tau_total_vals))/min(tau_total_vals)*100:.1f}%")
print("="*70)


GENERATING REIONIZATION HISTORY COMPARISON
✓ Saved: reionization_history_xe_boxsize_all
✓ Saved: reionization_history_xHI_boxsize_all

OPTICAL DEPTH CALCULATIONS

Physical constants:
  n_H0 = 2.523928e-07 cm^-3
  σ_T = 6.650000e-25 cm^2
  Prefactor = 5.179580e-07 Mpc^-1

BOX[Mpc]     Cell[kpc]    z_range         <ds>[Mpc]    τ_total   
---------------------------------------------------------------------------
400          3125         [5.00, 19.99] 3.128        0.036907  
800          6250         [5.00, 19.95] 6.263        0.040152  
1200         9375         [5.00, 19.99] 9.404        0.041276  
1600         12500        [5.00, 19.87] 12.552       0.041927  
2000         15625        [5.00, 19.87] 15.706       0.042363  

✓ Saved: tau_vs_z_boxsize_all
✓ Saved: dtau_vs_z_boxsize_all
✓ Saved: tau_total_vs_boxsize
✓ Saved: tau_total_vs_cellsize
✓ Saved: s_vs_z_boxsize

OPTICAL DEPTH ANALYSIS COMPLETE!
τ range: 0.036907 → 0.042363
Variation: 14.8%


In [7]:
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function for All Box Sizes
# kSZ integrand = (1 + δ) × x_e × v_z / c × e^(-τ(z))
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

# Speed of light in km/s
c_km_s = 299792.458  # km/s

print(f"Speed of light: c = {c_km_s:.6e} km/s")

# Dictionary to store kSZ results
kSZ_results = {}

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    if box_len not in tau_results:
        continue
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc (Cell = {box_len/HII_DIM_FIXED:.2f} Mpc)")
    print(f"{'='*70}")
    
    lightcone = lightcones[box_len]
    results = tau_results[box_len]
    
    # Extract redshift and distance axes (strip units)
    red_axis = np.asarray(results['red_axis'])
    z_mid = np.asarray(results['z_mid'])
    tau = np.asarray(results['tau'])
    
    # Extract 3D fields
    red_axis_full = np.asarray(lightcone.lightcone_redshifts)
    ind_z = np.where(red_axis_full <= z_max)[0]
    
    density_1plus = 1 + np.asarray(lightcone.density[:, :, ind_z])  # 1 + δ
    x_e_3D = 1 - np.asarray(lightcone.xH_box[:, :, ind_z])          # Ionized fraction
    v_los_km_s = np.asarray(lightcone.velocity[:, :, ind_z])        # Velocity [km/s]
    
    print(f"3D field shapes: {density_1plus.shape}")
    
    # =============================================================================
    # Interpolate τ(z) onto lightcone redshifts
    # =============================================================================
    
    tau_extended = np.concatenate([[0], tau])
    tau_at_lightcone = np.interp(red_axis, 
                                  np.concatenate([[red_axis[0]], z_mid]), 
                                  tau_extended)
    
    print(f"τ range: [{tau_at_lightcone.min():.6f}, {tau_at_lightcone.max():.6f}]")
    
    # Visibility function e^(-τ)
    visibility = np.exp(-tau_at_lightcone)
    print(f"e^(-τ) range: [{visibility.min():.6f}, {visibility.max():.6f}]")
    
    # Broadcast visibility to 3D
    visibility_3D = visibility[None, None, :]  # Shape (1, 1, n_redshift)
    
    # =============================================================================
    # Compute kSZ integrand WITH visibility function
    # =============================================================================
    
    kSZ_integrand = density_1plus * x_e_3D * v_los_km_s / c_km_s * visibility_3D
    
    print(f"\nkSZ INTEGRAND (with visibility) STATISTICS:")
    print(f"  Mean: {kSZ_integrand.mean():.4e}")
    print(f"  Std:  {kSZ_integrand.std():.4e}")
    print(f"  Min:  {kSZ_integrand.min():.4e}")
    print(f"  Max:  {kSZ_integrand.max():.4e}")
    print(f"  RMS:  {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")
    
    # Store results
    kSZ_results[box_len] = {
        'kSZ_integrand': kSZ_integrand,
        'visibility': visibility,
        'visibility_3D': visibility_3D,
        'tau_at_lightcone': tau_at_lightcone,
        'red_axis': red_axis,
        'ind_z': ind_z
    }
    
    # Add to lightcone object
    lightcone.kSZ_integrand = kSZ_integrand
    lightcone.visibility_func = visibility_3D

print("\n" + "="*70)
print("kSZ INTEGRAND CALCULATION COMPLETE")
print(f"Computed for {len(kSZ_results)} BOX_LEN values")
print("="*70)

# =============================================================================
# PLOT: kSZ Integrand - All Box Sizes Stacked
# =============================================================================

print("\n" + "="*70)
print("GENERATING kSZ INTEGRAND PLOTS")
print("="*70)

# Stacked plots for all box sizes
fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                         figsize=(14, 4*len(BOX_LEN_VALUES)), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

# Rainbow colormap for labels
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_results:
        ax.text(0.5, 0.5, f'BOX_LEN = {box_len:.0f} Mpc\nNo data', 
               ha='center', va='center',
               transform=ax.transAxes, fontsize=16, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        continue
        
    lightcone = lightcones[box_len]
    
    plotting.lightcone_sliceplot(lightcone, 'kSZ_integrand', ax=ax, fig=fig)
    
    # Change colormap to seismic (diverging colormap for positive/negative)
    im = ax.images[0]
    im.set_cmap('seismic')
    
    # Set symmetric color limits
    kSZ_data = kSZ_results[box_len]['kSZ_integrand']
    vmax = np.percentile(np.abs(kSZ_data), 99)
    im.set_clim(-vmax, vmax)
    
    # Get rainbow color for label
    color = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    # Add label with box info
    ax.text(0.02, 0.98, 
           f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)', 
           transform=ax.transAxes, 
           fontsize=13, fontweight='bold',
           verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.7, 
                    edgecolor='black', linewidth=2))
    
    # Add box size on right
    ax.text(0.98, 0.98, 
           f'{box_len:.0f} Mpc', 
           transform=ax.transAxes, fontsize=14, fontweight='bold',
           verticalalignment='top', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Save
plot_name = "kSZ_integrand_with_visibility_boxsize_stack"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

fig.suptitle(r'kSZ Integrand: $(1+\delta) \times x_e \times v_z/c \times e^{-\tau(z)}$ (HII_DIM=' + f'{HII_DIM_FIXED})', 
             fontsize=20, fontweight='bold', y=0.995)
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Visibility Function e^(-τ) vs z (All Box Sizes - Rainbow)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(kSZ_results.keys()):
    results = kSZ_results[box_len]
    red_axis_plot = results['red_axis']
    visibility_plot = results['visibility']
    
    color = cmap(norm(box_len))
    
    ax.plot(red_axis_plot, visibility_plot, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Visibility Function $e^{-\tau(z)}$', fontsize=20)
ax.set_ylim(0, 1.05)
ax.invert_xaxis()
ax.legend(fontsize=12, loc='best', ncol=1)
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "visibility_function_vs_z_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Visibility Function $e^{-\tau(z)}$ vs Redshift', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# Summary Statistics Table
# =============================================================================

print("\n" + "="*70)
print("kSZ INTEGRAND SUMMARY STATISTICS")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'Mean':<12} {'Std':<12} {'RMS':<12} {'Max|val|':<12}")
print("-" * 85)

for box_len in sorted(kSZ_results.keys()):
    results = kSZ_results[box_len]
    kSZ_int = results['kSZ_integrand']
    
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    mean_val = kSZ_int.mean()
    std_val = kSZ_int.std()
    rms_val = np.sqrt(np.mean(kSZ_int**2))
    max_abs_val = np.max(np.abs(kSZ_int))
    
    print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} {mean_val:<12.4e} {std_val:<12.4e} "
          f"{rms_val:<12.4e} {max_abs_val:<12.4e}")

print("\n" + "="*70)
print("kSZ INTEGRAND PLOTTING COMPLETE!")
print("="*70)


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION
Speed of light: c = 2.997925e+05 km/s

BOX_LEN = 400 Mpc (Cell = 3.12 Mpc)
3D field shapes: (128, 128, 963)
τ range: [0.000000, 0.036907]
e^(-τ) range: [0.963766, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 5.3988e-26
  Std:  7.2159e-23
  Min:  -1.3049e-21
  Max:  1.6812e-21
  RMS:  7.2159e-23

BOX_LEN = 800 Mpc (Cell = 6.25 Mpc)
3D field shapes: (128, 128, 481)
τ range: [0.000000, 0.040152]
e^(-τ) range: [0.960644, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 5.3997e-26
  Std:  7.0049e-23
  Min:  -7.7120e-22
  Max:  8.7806e-22
  RMS:  7.0049e-23

BOX_LEN = 1200 Mpc (Cell = 9.38 Mpc)
3D field shapes: (128, 128, 321)
τ range: [0.000000, 0.041276]
e^(-τ) range: [0.959564, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: -4.2535e-26
  Std:  6.6744e-23
  Min:  -7.0166e-22
  Max:  7.2349e-22
  RMS:  6.6744e-23

BOX_LEN = 1600 Mpc (Cell = 12.50 Mpc)
3D field shapes: (128, 128, 240)
τ range: [

In [8]:
# =============================================================================
# CELL 6: Compute Line-of-Sight Integrated kSZ Maps for All Box Sizes
# kSZ(z=5) = ∫ n_e0 σ_T (1/a²) (1+δ) x_e v_z/c e^(-τ) ds
# Integration from z=20 to z=5 along line of sight
# =============================================================================

print("\n" + "="*70)
print("LINE-OF-SIGHT kSZ MAP INTEGRATION")
print("="*70)

# Physical constants in CGS
print(f"\n=== PHYSICAL CONSTANTS (CGS) ===")
c_cm_s = 3.0e10  # cm/s
sigma_T_cm2 = 6.6525e-25  # cm²
n_e0_cm3 = 2.06e-7  # cm⁻³
Mpc_to_cm = 3.0857e24  # cm/Mpc

print(f"c = {c_cm_s:.2e} cm/s")
print(f"σ_T = {sigma_T_cm2:.4e} cm²")
print(f"n_e0 = {n_e0_cm3:.4e} cm⁻³")
print(f"1 Mpc = {Mpc_to_cm:.4e} cm")

# Calculate dimensionless prefactor
prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s  # [1/s]
print(f"\nPrefactor n_e0 × σ_T × c = {prefactor_cgs:.4e} s⁻¹")

# Dictionary to store kSZ map results
kSZ_map_results = {}

for box_len in BOX_LEN_VALUES:
    if box_len not in kSZ_results or box_len not in tau_results:
        continue
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc (Cell = {box_len/HII_DIM_FIXED:.2f} Mpc)")
    print(f"{'='*70}")
    
    lightcone = lightcones[box_len]
    results = kSZ_results[box_len]
    
    # Extract arrays
    red_axis = np.asarray(results['red_axis'])
    kSZ_integrand_with_vis = results['kSZ_integrand']
    
    # Get ds from tau_results
    ds_raw = tau_results[box_len]['ds']
    ds_Mpc = np.asarray(ds_raw)
    
    print(f"ds range: {ds_Mpc.min():.6f} - {ds_Mpc.max():.6f} Mpc")
    
    # Convert ds from Mpc to cm
    ds_cm = ds_Mpc * Mpc_to_cm  # cm
    print(f"ds in cm: {ds_cm.min():.4e} - {ds_cm.max():.4e} cm")
    
    # =============================================================================
    # Prepare integrand
    # =============================================================================
    
    # Scale factor a = 1/(1+z)
    a = 1.0 / (1.0 + red_axis)
    a_squared = a**2
    
    # Midpoint kSZ integrand
    kSZ_integrand_mid = 0.5 * (kSZ_integrand_with_vis[:, :, :-1] + 
                                kSZ_integrand_with_vis[:, :, 1:])
    
    # Midpoint a²
    a_squared_mid = 0.5 * (a_squared[:-1] + a_squared[1:])
    a_squared_mid_3D = a_squared_mid[None, None, :]
    
    # Full dimensionless integrand
    kSZ_integrand_full = (prefactor_cgs / a_squared_mid_3D) * kSZ_integrand_mid * (ds_cm / c_cm_s)[None, None, :]
    
    print(f"\nIntegrand shape: {kSZ_integrand_full.shape}")
    print(f"Integrating over {kSZ_integrand_full.shape[2]} redshift slices")
    print(f"From z = {red_axis.max():.2f} to z = {red_axis.min():.2f}")
    
    # =============================================================================
    # Integrate along line of sight
    # =============================================================================
    
    kSZ_map = np.sum(kSZ_integrand_full, axis=2)  # Shape (HII_DIM, HII_DIM), dimensionless
    
    print(f"\n=== kSZ MAP STATISTICS (DIMENSIONLESS) ===")
    print(f"Shape: {kSZ_map.shape}")
    print(f"Mean: {kSZ_map.mean():.4e}")
    print(f"Std:  {kSZ_map.std():.4e}")
    print(f"Min:  {kSZ_map.min():.4e}")
    print(f"Max:  {kSZ_map.max():.4e}")
    print(f"RMS:  {np.sqrt(np.mean(kSZ_map**2)):.4e}")
    
    # Store results
    kSZ_map_results[box_len] = {
        'kSZ_map': kSZ_map,
        'kSZ_integrand_full': kSZ_integrand_full
    }
    
    # Add to lightcone object
    lightcone.kSZ_map = kSZ_map

print("\n" + "="*70)
print("kSZ MAP INTEGRATION COMPLETE")
print(f"Computed maps for {len(kSZ_map_results)} BOX_LEN values")
print("="*70)

# =============================================================================
# PLOT: kSZ Maps - All Box Sizes Side-by-Side
# =============================================================================

print("\n" + "="*70)
print("GENERATING kSZ MAP PLOTS")
print("="*70)

fig, axes = plt.subplots(1, len(BOX_LEN_VALUES), 
                         figsize=(5*len(BOX_LEN_VALUES), 5.5), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

# Rainbow colormap for labels
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# Find global vmax for consistent color scale
vmax_global = 0
for box_len in kSZ_map_results.keys():
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    vmax_global = max(vmax_global, np.percentile(np.abs(kSZ_map), 99))

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_map_results:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
               transform=ax.transAxes, fontsize=16)
        ax.set_title(f'BOX={box_len:.0f} Mpc', fontsize=14)
        continue
        
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # Plot the kSZ map
    im = ax.imshow(kSZ_map.T,
                   cmap='seismic',
                   origin='lower',
                   extent=[0, box_len, 0, box_len],
                   aspect='equal',
                   vmin=-vmax_global,
                   vmax=vmax_global)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)
    
    # Labels
    ax.set_xlabel('x [Mpc]', fontsize=11)
    ax.set_ylabel('y [Mpc]', fontsize=11)
    
    # Get rainbow color for title
    color_label = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    ax.set_title(f'BOX={box_len:.0f} Mpc\nCell={cell_size_mpc:.2f} Mpc\n({cell_size_kpc:.0f} kpc)', 
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.6, 
                         edgecolor='black', linewidth=1.5))

# Overall title
fig.suptitle(r'kSZ Maps at $z=5$ (line-of-sight integrated, HII_DIM=' + f'{HII_DIM_FIXED})', 
             fontsize=18, fontweight='bold')

# Save
plot_name = "kSZ_maps_z5_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ Maps - Stacked Vertically
# =============================================================================

fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                         figsize=(8, 6*len(BOX_LEN_VALUES)), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_map_results:
        ax.text(0.5, 0.5, f'BOX={box_len:.0f} Mpc\nNo data', 
               ha='center', va='center',
               transform=ax.transAxes, fontsize=16, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        continue
        
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # Plot the kSZ map
    im = ax.imshow(kSZ_map.T,
                   cmap='seismic',
                   origin='lower',
                   extent=[0, box_len, 0, box_len],
                   aspect='equal',
                   vmin=-vmax_global,
                   vmax=vmax_global)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('kSZ (dimensionless)', fontsize=11)
    
    # Labels
    ax.set_xlabel('x [Mpc]', fontsize=12)
    ax.set_ylabel('y [Mpc]', fontsize=12)
    
    # Get rainbow color for label
    color_label = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    # Add label
    ax.text(0.02, 0.98, 
           f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)', 
           transform=ax.transAxes, fontsize=12, fontweight='bold',
           verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.7, 
                    edgecolor='black', linewidth=2))

# Overall title
fig.suptitle(r'kSZ Maps at $z=5$ - Box Size Comparison', 
             fontsize=20, fontweight='bold', y=0.995)

# Save
plot_name = "kSZ_maps_z5_boxsize_stack"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ Map Histograms - All Box Sizes Overlay
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(kSZ_map_results.keys()):
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    color = cmap(norm(box_len))
    
    ax.hist(kSZ_map.flatten(), bins=100, 
           color=color, 
           alpha=0.5, 
           edgecolor='black',
           linewidth=0.5,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('kSZ Signal (dimensionless)', fontsize=20)
ax.set_ylabel('Number of Pixels', fontsize=20)
ax.axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax.legend(fontsize=14, loc='best')
ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

# Save
plot_name = "kSZ_map_histogram_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'kSZ Map Pixel Distribution (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ RMS vs Box Size
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

box_vals = sorted(kSZ_map_results.keys())
rms_vals = []
std_vals = []
cell_sizes_kpc = []

for box_len in box_vals:
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    rms_vals.append(np.sqrt(np.mean(kSZ_map**2)))
    std_vals.append(np.std(kSZ_map))
    cell_sizes_kpc.append((box_len / HII_DIM_FIXED) * 1000)

ax.plot(box_vals, rms_vals, 'o-', linewidth=3, markersize=10,
       color='darkblue', label='RMS')
ax.plot(box_vals, std_vals, 's-', linewidth=3, markersize=10,
       color='darkred', label='Std Dev')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel('kSZ Signal (dimensionless)', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "kSZ_rms_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title('kSZ Map RMS vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# Summary Statistics Table
# =============================================================================

print("\n" + "="*70)
print("kSZ MAP SUMMARY STATISTICS")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'Mean':<12} {'Std':<12} {'RMS':<12} {'Max|val|':<12}")
print("-" * 85)

for box_len in sorted(kSZ_map_results.keys()):
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    mean_val = kSZ_map.mean()
    std_val = kSZ_map.std()
    rms_val = np.sqrt(np.mean(kSZ_map**2))
    max_abs_val = np.max(np.abs(kSZ_map))
    
    print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} {mean_val:<12.4e} {std_val:<12.4e} "
          f"{rms_val:<12.4e} {max_abs_val:<12.4e}")

print("\n" + "="*70)
print("kSZ MAP GENERATION COMPLETE!")
print("="*70)


LINE-OF-SIGHT kSZ MAP INTEGRATION

=== PHYSICAL CONSTANTS (CGS) ===
c = 3.00e+10 cm/s
σ_T = 6.6525e-25 cm²
n_e0 = 2.0600e-07 cm⁻³
1 Mpc = 3.0857e+24 cm

Prefactor n_e0 × σ_T × c = 4.1112e-21 s⁻¹

BOX_LEN = 400 Mpc (Cell = 3.12 Mpc)
ds range: 3.128242 - 3.128242 Mpc
ds in cm: 9.6528e+24 - 9.6528e+24 cm

Integrand shape: (128, 128, 962)
Integrating over 962 redshift slices
From z = 19.99 to z = 5.00

=== kSZ MAP STATISTICS (DIMENSIONLESS) ===
Shape: (128, 128)
Mean: -6.0825e-27
Std:  4.8015e-25
Min:  -2.4421e-24
Max:  2.5608e-24
RMS:  4.8019e-25

BOX_LEN = 800 Mpc (Cell = 6.25 Mpc)
ds range: 6.262967 - 6.262967 Mpc
ds in cm: 1.9326e+25 - 1.9326e+25 cm

Integrand shape: (128, 128, 480)
Integrating over 480 redshift slices
From z = 19.95 to z = 5.00

=== kSZ MAP STATISTICS (DIMENSIONLESS) ===
Shape: (128, 128)
Mean: -1.7114e-27
Std:  3.1596e-25
Min:  -1.5270e-24
Max:  1.5513e-24
RMS:  3.1597e-25

BOX_LEN = 1200 Mpc (Cell = 9.38 Mpc)
ds range: 9.404115 - 9.404115 Mpc
ds in cm: 2.9018e+25 -

In [9]:
# =============================================================================
# CELL 7: Compute kSZ Power Spectrum - P(k), C_ℓ, and D_ℓ for All Box Sizes
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ POWER SPECTRA")
print("="*70)

# CMB temperature conversion
T_CMB_0_K = 2.725  # K
z_obs = 5.0
T_CMB_z5_K = T_CMB_0_K * (1 + z_obs)  # K
T_CMB_z5_uK = T_CMB_z5_K * 1e6  # μK

print(f"\n=== CMB TEMPERATURE ===")
print(f"T_CMB(z=0) = {T_CMB_0_K:.3f} K")
print(f"T_CMB(z=5) = {T_CMB_z5_K:.3f} K = {T_CMB_z5_uK:.2f} μK")

# Angular diameter distance at z=5
D_A_Mpc = 1300  # Mpc (comoving)
print(f"Angular diameter distance: D_A = {D_A_Mpc:.1f} Mpc")

# Dictionary to store power spectrum results
power_spectrum_results = {}

for box_len in BOX_LEN_VALUES:
    if box_len not in kSZ_map_results:
        continue
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc (Cell = {box_len/HII_DIM_FIXED:.2f} Mpc)")
    print(f"{'='*70}")
    
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # =============================================================================
    # 1. Map Properties
    # =============================================================================
    
    npix_side = kSZ_map.shape[0]
    box_size_Mpc = box_len
    pix_size_Mpc = box_size_Mpc / npix_side
    
    print(f"\n=== MAP PROPERTIES ===")
    print(f"Map size: {npix_side} × {npix_side} pixels")
    print(f"Physical size: {box_size_Mpc:.1f} × {box_size_Mpc:.1f} Mpc²")
    print(f"Pixel size: {pix_size_Mpc:.3f} Mpc/pixel")
    print(f"Map RMS: {np.std(kSZ_map):.4e}")
    
    # Remove mean
    kSZ_map_centered = kSZ_map - np.mean(kSZ_map)
    print(f"Mean subtracted: {np.mean(kSZ_map_centered):.4e}")
    
    # =============================================================================
    # 2. Compute 2D Power Spectrum P(k)
    # =============================================================================
    
    # FFT and shift to center
    fft_map = np.fft.fft2(kSZ_map_centered)
    fft_map_shifted = np.fft.fftshift(fft_map)
    
    # Pixel area in physical units
    pix_area = pix_size_Mpc**2  # Mpc²
    
    # 2D power spectrum: P(k) in [Mpc²]
    ps2d = np.abs(fft_map_shifted)**2 * pix_area / (npix_side**4)
    
    print(f"\n=== 2D POWER SPECTRUM ===")
    print(f"P(k) range: [{ps2d.min():.4e}, {ps2d.max():.4e}] Mpc²")
    
    # =============================================================================
    # 3. k-space grid
    # =============================================================================
    
    # Fundamental frequency
    dk = 2 * np.pi / (npix_side * pix_size_Mpc)  # Mpc⁻¹
    
    # k-space coordinates
    kx = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    ky = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    kgrid = np.sqrt(kx[:, None]**2 + ky[None, :]**2)
    
    print(f"\n=== k-SPACE GRID ===")
    print(f"dk (fundamental): {dk:.6f} Mpc⁻¹")
    print(f"k range: [{kgrid.min():.6f}, {kgrid.max():.6f}] Mpc⁻¹")
    
    # =============================================================================
    # 4. Azimuthally Averaged P(k)
    # =============================================================================
    
    # Define k bins (logarithmic spacing)
    k_bins = np.logspace(np.log10(dk), np.log10(kgrid.max()*0.9), 35)
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])
    P1d = np.zeros(len(k_centers))
    P1d_err = np.zeros(len(k_centers))
    
    for i in range(len(k_centers)):
        mask = (kgrid >= k_bins[i]) & (kgrid < k_bins[i+1])
        if np.sum(mask) > 0:
            values = ps2d[mask]
            P1d[i] = np.mean(values)
            P1d_err[i] = np.std(values) / np.sqrt(np.sum(mask))
        else:
            P1d[i] = np.nan
    
    print(f"\n=== AZIMUTHALLY AVERAGED P(k) ===")
    valid_bins = np.sum(~np.isnan(P1d) & (P1d > 0))
    print(f"Valid k bins: {valid_bins} out of {len(k_centers)}")
    print(f"P(k) range: [{np.nanmin(P1d[P1d>0]):.4e}, {np.nanmax(P1d):.4e}] Mpc²")
    
    # =============================================================================
    # 5. Convert to ℓ space
    # =============================================================================
    
    # Convert k to ℓ
    ell_from_k = k_centers * D_A_Mpc
    
    # C_ℓ from P(k): C_ℓ = P(k) / D_A²
    Cl_approx = P1d / D_A_Mpc**2
    
    # D_ℓ = ℓ(ℓ+1) C_ℓ / 2π (dimensionless)
    Dl_dimensionless = ell_from_k * (ell_from_k + 1) * Cl_approx / (2 * np.pi)
    
    # D_ℓ in μK²
    Dl_uK2 = Dl_dimensionless * T_CMB_z5_uK**2
    
    print(f"\n=== D_ℓ CONVERSION ===")
    print(f"Conversion factor: T_CMB²(z=5) = {T_CMB_z5_uK**2:.4e} μK²")
    valid_dl = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0)
    if np.any(valid_dl):
        print(f"D_ℓ range: [{np.nanmin(Dl_uK2[valid_dl]):.4e}, {np.nanmax(Dl_uK2[valid_dl]):.4e}] μK²")
    
    # Store results
    power_spectrum_results[box_len] = {
        'k_centers': k_centers,
        'P1d': P1d,
        'P1d_err': P1d_err,
        'ell_from_k': ell_from_k,
        'Cl_approx': Cl_approx,
        'Dl_dimensionless': Dl_dimensionless,
        'Dl_uK2': Dl_uK2
    }

print("\n" + "="*70)
print("POWER SPECTRUM CALCULATION COMPLETE")
print(f"Computed for {len(power_spectrum_results)} BOX_LEN values")
print("="*70)

# =============================================================================
# PLOT 1: P(k) - All Box Sizes with Rainbow Colorbar
# =============================================================================

print("\n" + "="*70)
print("GENERATING POWER SPECTRUM PLOTS")
print("="*70)

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    k_centers = results['k_centers']
    P1d = results['P1d']
    
    valid = ~np.isnan(P1d) & (P1d > 0)
    color = cmap(norm(box_len))
    
    ax.loglog(k_centers[valid], P1d[valid], 
             color=color, linewidth=2.5, alpha=0.8,
             marker='o', markersize=4,
             label=f'{box_len:.0f} Mpc')

ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=20)
ax.set_ylabel(r'$P(k)$ [Mpc$^{2}$]', fontsize=20)
ax.legend(fontsize=12, loc='best', ncol=1)
ax.grid(True, alpha=0.3, which='both', linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "kSZ_Pk_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'kSZ Power Spectrum P(k) (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 2: D_ℓ in μK² - THE MAIN RAINBOW RESULT!
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    ell_from_k = results['ell_from_k']
    Dl_uK2 = results['Dl_uK2']
    
    valid = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0) & (ell_from_k > 10)
    color = cmap(norm(box_len))
    
    ax.loglog(ell_from_k[valid], Dl_uK2[valid], 
             color=color, linewidth=3.0, alpha=0.85,
             marker='s', markersize=5, zorder=10,
             label=f'{box_len:.0f} Mpc')

ax.set_xlabel(r'Multipole $\ell$', fontsize=20)
ax.set_ylabel(r'$D_\ell$ [$\mu$K$^2$]', fontsize=20)
ax.legend(fontsize=12, loc='best', ncol=1, framealpha=0.9)
ax.grid(True, alpha=0.3, which='both', linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "kSZ_Dl_uK2_boxsize_all_RAINBOW"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'kSZ Angular Power $D_\ell$ in μK² (HII_DIM=' + f'{HII_DIM_FIXED})', 
            fontsize=18, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name} *** MAIN RAINBOW PLOT ***")
plt.close(fig)

# =============================================================================
# PLOT 3: Peak D_ℓ vs BOX_LEN - KEY DIAGNOSTIC PLOT!
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

box_vals = sorted(power_spectrum_results.keys())
peak_Dl_vals = []
peak_ell_vals = []

for box_len in box_vals:
    results = power_spectrum_results[box_len]
    Dl_uK2 = results['Dl_uK2']
    ell_from_k = results['ell_from_k']
    valid = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0)
    
    if np.any(valid):
        peak_idx = np.argmax(Dl_uK2[valid])
        peak_Dl_vals.append(Dl_uK2[valid][peak_idx])
        peak_ell_vals.append(ell_from_k[valid][peak_idx])
    else:
        peak_Dl_vals.append(np.nan)
        peak_ell_vals.append(np.nan)

ax.plot(box_vals, peak_Dl_vals, 'o-', linewidth=3, markersize=10,
       color='darkred', label=r'Peak $D_\ell$')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel(r'Peak $D_\ell$ [$\mu$K$^2$]', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "kSZ_peak_Dl_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Peak kSZ Power vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 4: Peak ℓ vs BOX_LEN - Scale Diagnostic
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

ax.plot(box_vals, peak_ell_vals, 's-', linewidth=3, markersize=10,
       color='darkblue', label=r'$\ell$ at peak $D_\ell$')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel(r'Multipole $\ell$ at peak', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "kSZ_peak_ell_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Scale of Peak kSZ Power vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 5: Peak D_ℓ vs Cell Size
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

cell_sizes_kpc = [(b / HII_DIM_FIXED) * 1000 for b in box_vals]

ax.plot(cell_sizes_kpc, peak_Dl_vals, 'd-', linewidth=3, markersize=10,
       color='darkgreen', label=r'Peak $D_\ell$')

ax.set_xlabel(r'Cell Size [kpc]', fontsize=20)
ax.set_ylabel(r'Peak $D_\ell$ [$\mu$K$^2$]', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "kSZ_peak_Dl_vs_cellsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Peak kSZ Power vs Cell Size (Resolution)', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 6: ℓ Coverage Comparison
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    ell_from_k = results['ell_from_k']
    Dl_uK2 = results['Dl_uK2']
    
    valid = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0)
    
    if np.any(valid):
        ell_min = ell_from_k[valid].min()
        ell_max = ell_from_k[valid].max()
        
        color = cmap(norm(box_len))
        ax.axhspan(ell_min, ell_max, alpha=0.3, color=color, 
                  label=f'{box_len:.0f} Mpc')

ax.set_ylabel(r'Multipole $\ell$ Coverage', fontsize=20)
ax.set_xlabel('Box Size', fontsize=20)
ax.set_yscale('log')
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3, which='both', linestyle='--')
ax.set_xlim(-0.5, 0.5)
ax.set_xticks([])

plot_name = "kSZ_ell_coverage_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Multipole $\ell$ Coverage vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 7: Fundamental Mode (k_min) vs Box Size
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

k_min_vals = []
ell_min_vals = []

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    k_centers = results['k_centers']
    ell_from_k = results['ell_from_k']
    
    k_min_vals.append(k_centers[0])
    ell_min_vals.append(ell_from_k[0])

# Plot on twin axes
ax2 = ax.twinx()

line1 = ax.plot(box_vals, k_min_vals, 'o-', linewidth=3, markersize=10,
               color='purple', label=r'$k_{\rm min}$')
line2 = ax2.plot(box_vals, ell_min_vals, 's-', linewidth=3, markersize=10,
                color='orange', label=r'$\ell_{\rm min}$')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel(r'$k_{\rm min}$ [Mpc$^{-1}$]', fontsize=20, color='purple')
ax2.set_ylabel(r'$\ell_{\rm min}$', fontsize=20, color='orange')

ax.tick_params(axis='y', labelcolor='purple')
ax2.tick_params(axis='y', labelcolor='orange')

ax.grid(True, alpha=0.3, linestyle='--')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, fontsize=16, loc='best')

plot_name = "kSZ_fundamental_mode_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Fundamental Mode vs Box Size (Largest Scale)', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# Summary Table
# =============================================================================

print("\n" + "="*70)
print("POWER SPECTRUM SUMMARY")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'ℓ_range':<25} {'Peak_Dl[μK²]':<15} {'ℓ_peak':<10}")
print("-" * 85)

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    valid = ~np.isnan(results['Dl_uK2']) & (results['Dl_uK2'] > 0) & (results['ell_from_k'] > 10)
    
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    
    if np.any(valid):
        ell_min = results['ell_from_k'][valid].min()
        ell_max = results['ell_from_k'][valid].max()
        peak_dl = results['Dl_uK2'][valid].max()
        peak_idx = np.argmax(results['Dl_uK2'][valid])
        ell_peak = results['ell_from_k'][valid][peak_idx]
        
        print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} "
              f"[{ell_min:.1f}, {ell_max:.1f}]         "
              f"{peak_dl:<15.4e} {ell_peak:<10.1f}")

print(f"\n=== NOTES ===")
print(f"1. Fixed resolution: HII_DIM = {HII_DIM_FIXED}")
print(f"2. Cell size scales with box: Cell = BOX_LEN / HII_DIM")
print(f"3. Larger boxes probe larger scales (smaller ℓ)")
print(f"4. Uses flat-sky approximation: ℓ ≈ k × D_A")
print(f"5. D_A = {D_A_Mpc:.1f} Mpc at z={z_obs}")
print(f"6. Conversion: D_ℓ(μK²) = D_ℓ(dimensionless) × T_CMB²(z=5)")

print("\n" + "="*70)
print("ALL kSZ POWER SPECTRUM ANALYSIS COMPLETE!")
print("="*70)


COMPUTING kSZ POWER SPECTRA

=== CMB TEMPERATURE ===
T_CMB(z=0) = 2.725 K
T_CMB(z=5) = 16.350 K = 16350000.00 μK
Angular diameter distance: D_A = 1300.0 Mpc

BOX_LEN = 400 Mpc (Cell = 3.12 Mpc)

=== MAP PROPERTIES ===
Map size: 128 × 128 pixels
Physical size: 400.0 × 400.0 Mpc²
Pixel size: 3.125 Mpc/pixel
Map RMS: 4.8015e-25
Mean subtracted: 7.1746e-42

=== 2D POWER SPECTRUM ===
P(k) range: [4.5242e-83, 2.1745e-50] Mpc²

=== k-SPACE GRID ===
dk (fundamental): 0.015708 Mpc⁻¹
k range: [0.000000, 1.421723] Mpc⁻¹

=== AZIMUTHALLY AVERAGED P(k) ===
Valid k bins: 30 out of 34
P(k) range: [1.4305e-54, 7.7446e-51] Mpc²

=== D_ℓ CONVERSION ===
Conversion factor: T_CMB²(z=5) = 2.6732e+14 μK²
D_ℓ range: [3.2893e-41, 2.8324e-39] μK²

BOX_LEN = 800 Mpc (Cell = 6.25 Mpc)

=== MAP PROPERTIES ===
Map size: 128 × 128 pixels
Physical size: 800.0 × 800.0 Mpc²
Pixel size: 6.250 Mpc/pixel
Map RMS: 3.1596e-25
Mean subtracted: 2.1524e-42

=== 2D POWER SPECTRUM ===
P(k) range: [1.9636e-82, 5.4443e-50] Mpc²

